In [1]:
import numpy as np
import pandas as pd
import math
import hashlib

In [2]:
import sys
import os

sys.path.insert(0, os.path.abspath("cascade-python"))

from cascade.key import Key
from cascade.mock_classical_channel import MockClassicalChannel
from cascade.reconciliation import Reconciliation

In [3]:
sys.set_int_max_str_digits(9000)

In [4]:
data_path = "mock_output.csv"

In [5]:
data = pd.read_csv(data_path)

In [6]:
data = data.rename(columns={"used_for_parameter_estimation" : "parameter"})
data = data.rename(columns={"bob_measured_quadrature" : "bob_basis"})

In [7]:
# in the real data we won't already have this so simulate we don't
data = data.drop(["alice_value_for_bob_basis"], axis=1)

### Keep only Alice values corresponding to Bob's measurements

In [8]:
# keep track of when bob measured X and when he measured P
bob_x_idxs = data["bob_basis"] == "X"
bob_p_idxs = data["bob_basis"] == "P"

In [9]:
alice_value_x = data[bob_x_idxs]["alice_x"]
alice_value_p = data[bob_p_idxs]["alice_p"]

In [10]:
alice_value = pd.concat([alice_value_x, alice_value_p])

In [11]:
data.insert(6, "alice_value", alice_value)

In [12]:
data.head()

,pulse_id,alice_x,alice_p,bob_basis,bob_value,parameter,alice_value
0,0,0.609434,-0.903902,X,-2.315105,False,0.609434
1,1,-2.079968,-1.331755,X,-2.134312,False,-2.079968
2,2,1.500902,0.868020,X,0.574552,False,1.500902
3,3,1.881129,0.503709,X,1.668453,False,1.881129
4,4,-3.902070,-2.809583,X,-2.570418,False,-3.902070


### Estimate information rate between Alice and Bob

In [13]:
# only use subset designated for parameter estimation
parameter_subset = data[data["parameter"] == True]

A = parameter_subset["alice_value"]
B = parameter_subset["bob_value"]

### Channel gain

### $g = \frac{\text{Cov}(A,B)}{\text{Var}(A)}$

In [14]:
channel_gain = np.cov(A, B)[0,1] / np.var(A)
print(f"Channel gain g : {channel_gain}")

Channel gain g : 0.8165714135341845


### Noise variance

Variance of the noise that remains after removing the part of Bob's measurement that is explained by Alice's value

#### $\sigma_n^2 = \text{Var}(B - gA)$

In [15]:
noise_var = np.var(B - channel_gain * A)
print(f"The noise variance is : {noise_var}")

The noise variance is : 1.056759005609136


### Signal to noise ratio

### $\Sigma_B = \frac{g^2 \text{Var}(A)}{\sigma_n^2} $

In [16]:
bob_snr = (channel_gain * np.var(A)) / noise_var
print(f"Bob's signal to noise ratio : {bob_snr}")

Bob's signal to noise ratio : 2.9271795225417563


### Information rate between Alice and Bob

### $I_{AB} = \frac{1}{2} \log_2 (1 + \Sigma_B)$

In [17]:
I_AB = 0.5 * (np.log2(1 + bob_snr))
print(f"I_AB = {I_AB}")

I_AB = 0.9867467746019641


### Upper bound on Eve's information

### Input noise

### $\chi = \frac{\text{Var}(n)}{g^2}$

In [18]:
chi = noise_var / (channel_gain**2)
print(f"Chi = {chi}")

Chi = 1.5848479897673238


### Eve signal to noise ratio

### $\Sigma_E = \frac{\text{Var}(A)}{1 + 1/\chi}$

In [19]:
eve_snr = np.var(A) / (1 + (1/chi))
print(f"Eve SNR : {eve_snr}")

Eve SNR : 2.322649882618737


### Information rate between Alice and Eve (upper bound)

Since we don't have Eve's data we cannot directly estimate it, however we can upper bound her SNR based on channel information estimated from the Alice and Bob data that we have, therefore we can upper bound her information rate

### $I_{AE} = \frac{1}{2} \log_2 (1 + \Sigma_E)$

In [20]:
I_AE = 0.5 * (np.log2(1 + eve_snr))
print(f"I_AE = {I_AE}")

I_AE = 0.8661671400116104


### Secret information rate

### $\Delta I = I_{AB} - I_{AE}$

For the protocol we require that $\Delta I > 0$

In [21]:
secret_rate = I_AB - I_AE
print(f"Secret information rate : {secret_rate}")

Secret information rate : 0.1205796345903537


## Converting gaussian variables to bitstrings

### Sliced reconciliation parameters

Each real value will become an m bit bitstring

In [22]:
# number of slices
m = 4
n_intervals = 2**m

In [23]:
A_vals = data["alice_value"]
B_vals = data["bob_value"]

In [24]:
def calculate_thresholds(A_vals, n_intervals):
    """Calculate tresholds for n intervals based on Gaussian quantiles
    Each resulting range has equal probability"""

    probs = np.linspace(0, 1, n_intervals + 1)
    thresholds = np.quantile(A_vals, probs)

    thresholds[0] = -np.inf
    thresholds[-1] = np.inf
    
    return thresholds

In [25]:
thresholds = calculate_thresholds(A_vals, n_intervals)
print(f"Calculated thresholds for {m} slices\n")
print(thresholds)

Calculated thresholds for 4 slices

[       -inf -3.11231869 -2.32424634 -1.77958578 -1.36074698 -0.9691056
 -0.61419933 -0.32544828 -0.01468415  0.26331113  0.60647753  0.91021914
  1.26601326  1.67738299  2.2101442   2.95894032         inf]


### Map each of Alice's values to an interval

In [26]:
def interval_indices(values, thresholds):
    """Return the index of the interval where each real value is"""
    
    return np.digitize(values, thresholds[1:-1], right=False)

In [27]:
A_intervals = [int(x) for x in interval_indices(A_vals, thresholds)]

B_intervals = [int(x) for x in interval_indices(B_vals, thresholds)]

print(f"Sample Alice values: {list(A_vals[:3])}")
print(f"Assigned intervals: {list(A_intervals[:3])}\n")

print(f"Sample Bob values: {list(B_vals[:3])}")
print(f"Assigned intervals: {list(B_intervals[:3])}")

Sample Alice values: [0.6094341595088627, -2.079968212480991, 1.5009023916129145]
Assigned intervals: [10, 2, 12]

Sample Bob values: [-2.315105067702726, -2.1343121604505857, 0.5745518907282836]
Assigned intervals: [2, 2, 9]


In [28]:
def interval_bitstrings(interval_idxs, m):
    
    bit_map = {idx: None for idx in interval_idxs}
    
    bitstr = ""
    
    for idx in interval_idxs:
        
        # calculate binary representation of idx padded to m bits
        bin_rep = bin(idx)[2:].zfill(m)
        
        assert len(bin_rep) == m
        
        bit_map[idx] = bin_rep
        bitstr += bin_rep
    
    return bitstr, bit_map

### Assign bitstring labels to intervals

In [29]:
A_bits, alice_bit_map = interval_bitstrings(A_intervals, m)

print(f"Alice bitstring : {A_bits[:20]} ... | total length {len(A_bits)}")

Alice bitstring : 10100010110011010000 ... | total length 8000


In [30]:
B_bits, bob_bit_map = interval_bitstrings(B_intervals, m)

print(f"Bob bitstring : {B_bits[:20]} ... | total length {len(B_bits)}")

Bob bitstring : 00100010100111000001 ... | total length 8000


In [31]:
def calculate_mismatch(A_bits, B_bits):

    return sum([int("0b" + A_bits[i], 2) ^ int("0b" + B_bits[i], 2) for i in range(len(B_bits))])

### Estimate error rate

In [32]:
errors = calculate_mismatch(A_bits, B_bits)

qber = errors / len(A_bits)

print(f"Number of different bits in Alice and Bob strings : {errors} out of {len(A_bits)} | QBER {qber}")

Number of different bits in Alice and Bob strings : 2940 out of 8000 | QBER 0.3675


### Output to files

In [33]:
with open("alice_key.txt", "w") as text_file:
    text_file.write(A_bits)

with open("bob_key.txt", "w") as text_file:
    text_file.write(B_bits)


### Cascade error correction

The original paper does it slice by slice while here we are directly correcting the whole key, but the correction succeeds and it shouldn't change the amount of leaked information so this shouldn't be an issue

In [34]:
# small helper function, as we need to convert the bitstrings to keyobjects
def key_from_bits(bitstring):
    key = Key()
    key._size = len(bitstring)

    for i, b in enumerate(bitstring):
        key._bits[i] = int(b)

    return key

In [35]:
correct_key = key_from_bits(A_bits)
noisy_key = key_from_bits(B_bits)

channel = MockClassicalChannel(correct_key)

algorithm = "original"

rec = Reconciliation(
    algorithm,
    channel,
    noisy_key,
    qber
)

rec.reconcile()
rec_key = rec.get_reconciled_key()

leakage = rec.stats.reply_parity_bits

B_bits = str(rec_key)

### Make sure their bits are actually the same after error correction

In [36]:
assert calculate_mismatch(A_bits, B_bits) == 0

In [37]:
shared_key = A_bits

### Privacy amplification

We remove the number of bits leaked by the error correction step, the bound on the number of bits assumed to be known by Eve from the original parameter estimation and a little extra to be sure

In [38]:
def calculate_final_length(shared_key, I_AE, cascade_leakage, security_margin=128):
    
    return len(shared_key) - math.ceil(I_AE * len(shared_key)) - cascade_leakage - security_margin

In [39]:
test_leakage = 500

final_key_len = calculate_final_length(shared_key, I_AE, test_leakage)

print(f"The final key length after privacy amplification will be: {final_key_len}")

The final key length after privacy amplification will be: 442


In [40]:
def toeplitz_hash(raw_bits, output_len, seed=None):
    
    raw_bits = np.asarray([int(x) for x in raw_bits])
    input_len = len(raw_bits)

    rng = np.random.default_rng(seed)

    toeplitz_seed = rng.integers(
        0, 2, size=output_len + input_len - 1, dtype=np.uint8
    )

    final_key = np.empty(output_len, dtype=np.uint8)

    for row in range(output_len):
        # Toeplitz row:
        # T[row, col] = seed[row - col + input_len - 1]
        row_bits = toeplitz_seed[
            row + input_len - 1 - np.arange(input_len)
        ]

        final_key[row] = np.bitwise_xor.reduce(row_bits & raw_bits)

    return final_key, toeplitz_seed

In [41]:
key_toeplitz, _ = toeplitz_hash(shared_key, final_key_len)

In [42]:
key_toeplitz = "".join([str(x) for x in key_toeplitz.tolist()])

In [43]:
print(f"Key after privacy amplification using Toeplitz hash function")
print(key_toeplitz)

Key after privacy amplification using Toeplitz hash function
0000110011100010111111111111011110100100011101010001111000000110011110010101111100010010011000101110110101011111100101011101000111011100000110101001001001001110111101011011000001111001101001100101101010011000000111010010110011001100100001101010111010110101110110100100011101110110101011110101111110010000111110100001110111010010110001111100111000011100001110010001010010101110111100011111110100000111001101110010110001011100011001011110011010


In [44]:
assert len(key_toeplitz) == final_key_len

In [45]:
def bitstr_to_bytes(bstr):
    
    assert len(bstr) % 8 == 0
    
    return bytes([int("0b" + bstr[i:i+8], 2) for i in range(0, len(bstr)-8)])

In [46]:
def sha256_hash(raw_bits, output_len, salt=None):

    raw_bytes = bitstr_to_bytes(raw_bits)

    if salt is None:
        salt = os.urandom(32)

    needed_bytes = math.ceil(output_len / 8)
    output = b""

    counter = 0
    while len(output) < needed_bytes:
        counter_bytes = counter.to_bytes(8, "big")
        digest = hashlib.sha256(salt + counter_bytes + raw_bytes).digest()
        output += digest
        counter += 1

    final_bytes = output[:needed_bytes]
    final_bits = np.unpackbits(np.frombuffer(final_bytes, dtype=np.uint8))

    return "".join([str(x) for x in final_bits[:output_len]]), salt

In [47]:
key_sha, _ = sha256_hash(shared_key, final_key_len)

In [48]:
print(f"Key after privacy amplification using SHA256 hash function")
print(key_sha)

Key after privacy amplification using SHA256 hash function
0100101000101010010100111101110001110001000001001110111111110010101111001101101011011001000010010111011001000011100011000100110100010101010011111110111100100000100011111110111010111101100110100100101111101111001100100010000110110000111110100000111011010100001011010001101111011101011001001000011011100010001100001101100110000111011110111001010100000111011010010100101010001010001101101101110101101111101000011001011001000110101111000011100101


In [49]:
assert len(key_sha) == final_key_len